In [1]:
import json
from pathlib import Path

ROOT = Path(r"D:\projects\f1-project")

for nb_path in sorted((ROOT / "notebooks").glob("*.ipynb")):
    notebook = json.loads(nb_path.read_text(encoding="utf-8"))
    for cell in notebook["cells"]:
        if cell["cell_type"] != "code":
            continue
        source = "".join(cell["source"])
        looks_like_fetching = ("def " in source) and (
            "requests" in source or "cache" in source.lower() or "jolpica" in source.lower()
        )
        if looks_like_fetching:
            print("=" * 25, nb_path.name)
            print(source)
            print()

========================= 01_race_data_exploration.ipynb
import requests
import pandas as pd
import time

BASE_URL = "https://api.jolpi.ca/ergast/f1"

def fetch_race_laps(season, round_num, limit=100):
    url = f"{BASE_URL}/{season}/{round_num}/laps/"
    offset = 0
    total = None
    rows = []

    while total is None or offset < total:
        params = {"limit": limit, "offset": offset}
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()

        mr_data = response.json()["MRData"]
        total = int(mr_data["total"])
        returned_limit = int(mr_data["limit"])
        returned_offset = int(mr_data["offset"])

        races = mr_data["RaceTable"].get("Races", [])
        if not races:
            break

        lap_groups = races[0].get("Laps", [])

        for lap_group in lap_groups:
            lap_number = int(lap_group["number"])

            for timing in lap_group.get("Timings", []):
                rows.append({
         

In [2]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import requests

ROOT = Path(r"D:\projects\f1-project")
RAW_DIR = ROOT / "data" / "raw" / "jolpica"
LAP_CACHE_DIR = RAW_DIR / "laps"
PIT_CACHE_DIR = RAW_DIR / "pit_stops"
PROCESSED_DIR = ROOT / "data" / "processed"
for folder in (LAP_CACHE_DIR, PIT_CACHE_DIR, PROCESSED_DIR):
    folder.mkdir(parents=True, exist_ok=True)

API = "https://api.jolpi.ca/ergast/f1"
SLEEP_SECONDS = 1.0          # pause after every real download
SEASONS = [2023, 2024]
KEYS = ["season", "round", "lap", "driver_id"]


def to_seconds(text):
    """'1:39.019' -> 99.019, '23.4' -> 23.4, None/NaN -> NaN."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return np.nan
    total = 0.0
    for part in str(text).split(":"):
        total = total * 60 + float(part)
    return total


def get_json(url, params, cache_file, max_retries=6):
    """Return the JSON for one API page. Uses the cache if present; otherwise downloads,
    retrying with growing waits if the API says 'too many requests' (429)."""
    if cache_file.exists():
        return json.loads(cache_file.read_text(encoding="utf-8"))

    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, timeout=30)
        except requests.RequestException as err:
            print(f"   network error ({err}); retrying")
            time.sleep(5 * 2 ** attempt)
            continue

        if response.status_code == 429:
            retry_after = response.headers.get("Retry-After", "")
            wait = float(retry_after) if retry_after.isdigit() else 5 * 2 ** attempt
            print(f"   429 rate limit; waiting {wait:.0f}s")
            time.sleep(wait)
            continue

        response.raise_for_status()
        data = response.json()
        cache_file.write_text(json.dumps(data), encoding="utf-8")
        time.sleep(SLEEP_SECONDS)
        return data

    raise RuntimeError(f"Still blocked after {max_retries} tries: {url}")


def get_schedule(season):
    data = get_json(f"{API}/{season}.json", {"limit": 100}, RAW_DIR / f"{season}_schedule.json")
    races = data["MRData"]["RaceTable"]["Races"]
    return pd.DataFrame([
        {"season": season, "round": int(r["round"]), "race_name": r["raceName"], "date": r["date"]}
        for r in races
    ])


def fetch_laps(season, rnd):
    """All lap timings for one race. Moves the offset by the page limit the API reports."""
    url = f"{API}/{season}/{rnd}/laps/"
    rows, offset, total = [], 0, 0
    while True:
        cache_file = LAP_CACHE_DIR / f"{season}_round_{rnd}_laps_offset_{offset}.json"
        data = get_json(url, {"limit": 100, "offset": offset}, cache_file)
        mr = data["MRData"]
        total = int(mr["total"])
        races = mr["RaceTable"].get("Races", [])
        if not races:
            break
        for lap_group in races[0].get("Laps", []):
            lap_number = int(lap_group["number"])
            for timing in lap_group.get("Timings", []):
                rows.append({
                    "season": season, "round": rnd, "lap": lap_number,
                    "driver_id": timing["driverId"],
                    "position": int(timing["position"]),
                    "lap_time": timing["time"],
                })
        offset = int(mr["offset"]) + int(mr["limit"])   # NOT len(lap groups)
        if offset >= total:
            break
    columns = ["season", "round", "lap", "driver_id", "position", "lap_time"]
    return pd.DataFrame(rows, columns=columns), total


def fetch_pit_stops(season, rnd):
    """All pit stops for one race. Keeps the RAW duration text; parsed later with to_seconds."""
    url = f"{API}/{season}/{rnd}/pitstops/"
    rows, offset = [], 0
    while True:
        name = (f"{season}_round_{rnd}_pit_stops.json" if offset == 0
                else f"{season}_round_{rnd}_pit_stops_offset_{offset}.json")
        data = get_json(url, {"limit": 100, "offset": offset}, PIT_CACHE_DIR / name)
        mr = data["MRData"]
        total = int(mr["total"])
        races = mr["RaceTable"].get("Races", [])
        for stop in (races[0].get("PitStops", []) if races else []):
            rows.append({
                "season": season, "round": rnd, "driver_id": stop["driverId"],
                "lap": int(stop["lap"]), "stop": int(stop["stop"]),
                "time": stop.get("time"), "duration_raw": stop.get("duration"),
            })
        offset = int(mr["offset"]) + int(mr["limit"])
        if offset >= total:
            break
    columns = ["season", "round", "driver_id", "lap", "stop", "time", "duration_raw"]
    return pd.DataFrame(rows, columns=columns)


# ---------- main loop ----------
schedule = pd.concat([get_schedule(s) for s in SEASONS], ignore_index=True)
print("Scheduled races:", len(schedule))

existing_laps = pd.read_csv(PROCESSED_DIR / "laps_with_pit_flags_2023_2024_first5.csv")
existing_laps = existing_laps[["season", "round", "lap", "driver_id", "position", "lap_time"]]
already_have = set(zip(existing_laps["season"], existing_laps["round"]))

lap_frames, pit_frames, qc_rows = [existing_laps], [], []
incomplete = False
t0 = time.time()

for race in schedule.itertuples():
    season, rnd = int(race.season), int(race.round)
    try:
        if (season, rnd) not in already_have:
            laps_df, api_total = fetch_laps(season, rnd)
            lap_frames.append(laps_df)
            qc_rows.append({
                "season": season, "round": rnd, "rows": len(laps_df), "api_total": api_total,
                "duplicates": int(laps_df.duplicated(KEYS).sum()),
                "laps": laps_df["lap"].max(), "drivers": laps_df["driver_id"].nunique(),
            })
        pit_frames.append(fetch_pit_stops(season, rnd))
    except RuntimeError as err:
        print("STOPPED:", err)
        incomplete = True
        break
    print(f"{season} R{rnd:>2} {race.race_name:<30} done  ({(time.time() - t0) / 60:.1f} min elapsed)")

# ---------- save and check ----------
laps_all = pd.concat(lap_frames, ignore_index=True)
pits_all = pd.concat(pit_frames, ignore_index=True)
pits_all["duration_seconds"] = pits_all["duration_raw"].apply(to_seconds)

laps_all.to_csv(PROCESSED_DIR / "laps_all_2023_2024.csv", index=False)
pits_all.to_csv(PROCESSED_DIR / "pit_stops_all_2023_2024.csv", index=False)

qc = pd.DataFrame(qc_rows)
qc.to_csv(PROCESSED_DIR / "fetch_qc_2023_2024.csv", index=False)

print("\nraces with laps:", laps_all.groupby(["season", "round"]).ngroups,
      "| lap rows:", len(laps_all),
      "| duplicate lap keys:", int(laps_all.duplicated(KEYS).sum()))
if len(qc):
    print("races where rows != API total:\n", qc[qc["rows"] != qc["api_total"]])
print("\npit stops:", len(pits_all),
      "| duration missing in the API:", int(pits_all["duration_raw"].isna().sum()),
      "| duration in m:ss format:", int(pits_all["duration_raw"].astype(str).str.contains(":").sum()),
      "| still unparsed:", int(pits_all["duration_seconds"].isna().sum()))
print("\n2023 R3 raw durations:",
      pits_all[(pits_all["season"] == 2023) & (pits_all["round"] == 3)]["duration_raw"].head(20).tolist())
if incomplete:
    print("\nIncomplete (rate-limited). Wait ~15 minutes, then run this cell again: it resumes from the cache.")

Scheduled races: 46
2023 R 1 Bahrain Grand Prix             done  (0.0 min elapsed)
2023 R 2 Saudi Arabian Grand Prix       done  (0.0 min elapsed)
2023 R 3 Australian Grand Prix          done  (0.1 min elapsed)
2023 R 4 Azerbaijan Grand Prix          done  (0.1 min elapsed)
2023 R 5 Miami Grand Prix               done  (0.2 min elapsed)
2023 R 6 Monaco Grand Prix              done  (1.0 min elapsed)
2023 R 7 Spanish Grand Prix             done  (1.8 min elapsed)
2023 R 8 Canadian Grand Prix            done  (2.7 min elapsed)
2023 R 9 Austrian Grand Prix            done  (3.5 min elapsed)
2023 R10 British Grand Prix             done  (4.0 min elapsed)
2023 R11 Hungarian Grand Prix           done  (4.7 min elapsed)
2023 R12 Belgian Grand Prix             done  (5.2 min elapsed)
2023 R13 Dutch Grand Prix               done  (6.1 min elapsed)
2023 R14 Italian Grand Prix             done  (6.6 min elapsed)
2023 R15 Singapore Grand Prix           done  (7.3 min elapsed)
2023 R16 Japanese Gr

ValueError: could not convert string to float: ''

In [3]:
# CELL 3 — INSPECT RAW PIT-STOP DURATIONS

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r"D:\projects\f1-project")
PROCESSED = ROOT / "data" / "processed"

# Reconstruct pit-stop data from the cached API responses
# First, load the CSV if it was successfully saved previously.
pit_file = PROCESSED / "pit_stops_all_2023_2024.csv"

if pit_file.exists():
    pits_all = pd.read_csv(pit_file)
    print("Loaded existing pit-stop CSV.")
else:
    print("Pit-stop CSV was not saved because the previous cell crashed.")
    print("We will reconstruct it from cached JSON in the next step.")

# Inspect malformed values if the CSV exists
if pit_file.exists() and "duration_raw" in pits_all.columns:

    durations = pits_all["duration_raw"].astype("string")

    invalid = (
        durations.isna()
        | durations.str.strip().eq("")
    )

    print("\nTotal pit-stop records:", len(pits_all))
    print("Missing or empty duration values:", invalid.sum())

    print("\nProblematic examples:")
    display(pits_all.loc[invalid].head(20))

else:
    print("\nNo existing CSV available for inspection.")

Pit-stop CSV was not saved because the previous cell crashed.
We will reconstruct it from cached JSON in the next step.

No existing CSV available for inspection.


In [4]:
import numpy as np
import pandas as pd

KEYS = ["season", "round", "lap", "driver_id"]


def to_seconds_safe(text):
    """'1:39.019' -> 99.019; '23.4' -> 23.4; None, '' or unreadable text -> NaN."""
    if text is None or (isinstance(text, float) and np.isnan(text)):
        return np.nan
    text = str(text).strip()
    if text == "":
        return np.nan
    try:
        total = 0.0
        for part in text.split(":"):
            total = total * 60 + float(part)
        return total
    except ValueError:
        return np.nan


laps_all = pd.concat(lap_frames, ignore_index=True)
pits_all = pd.concat(pit_frames, ignore_index=True)
pits_all["duration_seconds"] = pits_all["duration_raw"].apply(to_seconds_safe)

laps_all.to_csv(PROCESSED_DIR / "laps_all_2023_2024.csv", index=False)
pits_all.to_csv(PROCESSED_DIR / "pit_stops_all_2023_2024.csv", index=False)
qc = pd.DataFrame(qc_rows)
qc.to_csv(PROCESSED_DIR / "fetch_qc_2023_2024.csv", index=False)

# ---------- laps ----------
print("LAPS")
print("races:", laps_all.groupby(["season", "round"]).ngroups,
      "| rows:", len(laps_all),
      "| duplicate keys:", int(laps_all.duplicated(KEYS).sum()))
print("new races where rows != API total:", int((qc["rows"] != qc["api_total"]).sum()), "of", len(qc))
print("new races with duplicate keys:", int((qc["duplicates"] > 0).sum()))
per_race = laps_all.groupby(["season", "round"]).agg(rows=("lap", "size"), laps=("lap", "max"), drivers=("driver_id", "nunique"))
print(per_race.describe().loc[["min", "max"]])

# ---------- pit stops ----------
raw = pits_all["duration_raw"]
print("\nPIT STOPS")
print("stops:", len(pits_all),
      "| duplicate (race, driver, stop):", int(pits_all.duplicated(["season", "round", "driver_id", "stop"]).sum()))
print("duration is None in API:", int(raw.isna().sum()),
      "| empty string:", int((raw.astype(str).str.strip() == "").sum()),
      "| m:ss format:", int(raw.astype(str).str.contains(":").sum()),
      "| unparsed after cleaning:", int(pits_all["duration_seconds"].isna().sum()))

bad = pits_all[pits_all["duration_seconds"].isna()]
print("\nraw values that stayed unparsed:", bad["duration_raw"].astype(str).value_counts().head(5).to_dict())
print("unparsed by race (top 6):")
print(bad.groupby(["season", "round"]).size().sort_values(ascending=False).head(6))

# ---------- does the new pipeline agree with the old one for the first 10 races? ----------
old_pits = pd.read_csv(PROCESSED_DIR / "pit_stops_2023_2024_first5.csv")
first10 = pits_all[[(s, r) in already_have for s, r in zip(pits_all["season"], pits_all["round"])]]
print("\nfirst 10 races | stops: old", len(old_pits), "new", len(first10),
      "| missing durations: old", int(old_pits["duration"].isna().sum()),
      "new", int(first10["duration_seconds"].isna().sum()))
print("2023 R3 raw durations:", pits_all[(pits_all["season"] == 2023) & (pits_all["round"] == 3)]["duration_raw"].head(12).tolist())

LAPS
races: 46 | rows: 50960 | duplicate keys: 0
new races where rows != API total: 0 of 36
new races with duplicate keys: 0
       rows  laps  drivers
min   815.0  44.0     16.0
max  1513.0  78.0     20.0

PIT STOPS
stops: 1737 | duplicate (race, driver, stop): 0
duration is None in API: 0 | empty string: 1 | m:ss format: 159 | unparsed after cleaning: 1

raw values that stayed unparsed: {'': 1}
unparsed by race (top 6):
season  round
2024    12       1
dtype: int64

first 10 races | stops: old 374 new 374 | missing durations: old 70 new 0
2023 R3 raw durations: ['18.056', '18.951', '18.382', '17.657', '21.659', '19.158', '17.814', '19.146', '18.319', '20.789', '15:25.181', '15:25.112']


In [5]:
# CELL 4 — RECOVER DATA AND SAVE SAFELY

import pandas as pd
import numpy as np
import json

# 1. Verify that the original fetch results still exist
required_variables = ["lap_frames", "pit_frames", "qc"]

for name in required_variables:
    if name not in globals():
        raise NameError(
            f"{name} is missing. Do not rerun the API fetch yet."
        )

print("Lap frames:", len(lap_frames))
print("Pit-stop frames:", len(pit_frames))
print("QC records:", len(qc))

# 2. Reconstruct combined datasets
laps_all = pd.concat(lap_frames, ignore_index=True)
pits_all = pd.concat(pit_frames, ignore_index=True)

# 3. Robust duration conversion
def safe_to_seconds(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip()

    if text == "":
        return np.nan

    try:
        parts = text.split(":")
        total = 0.0

        for part in parts:
            total = total * 60 + float(part)

        return total

    except (ValueError, TypeError):
        return np.nan

# Preserve original duration text
pits_all["duration_seconds"] = (
    pits_all["duration_raw"].apply(safe_to_seconds)
)

# 4. Save all datasets
laps_all.to_csv(
    PROCESSED_DIR / "laps_all_2023_2024.csv",
    index=False
)

pits_all.to_csv(
    PROCESSED_DIR / "pit_stops_all_2023_2024.csv",
    index=False
)

qc_df = pd.DataFrame(qc_rows)
qc_df.to_csv(
    PROCESSED_DIR / "fetch_qc_2023_2024.csv",
    index=False
)

# 5. Report results
print("\nLAP DATA")
print("-" * 45)
print("Rows:", len(laps_all))
print("Unique races:", laps_all.groupby(["season", "round"]).ngroups)
print("Duplicate lap keys:",
      laps_all.duplicated(KEYS).sum())

print("\nPIT-STOP DATA")
print("-" * 45)
print("Records:", len(pits_all))
print("Unique races:", pits_all.groupby(["season", "round"]).ngroups)
print("Missing raw durations:", pits_all["duration_raw"].isna().sum())
print("Empty raw durations:",
      pits_all["duration_raw"].astype("string").str.strip().eq("").sum())
print("Unparsed durations:", pits_all["duration_seconds"].isna().sum())

print("\nSAVED FILES")
print("-" * 45)
print("laps_all_2023_2024.csv")
print("pit_stops_all_2023_2024.csv")
print("fetch_qc_2023_2024.csv")

Lap frames: 37
Pit-stop frames: 46
QC records: 36

LAP DATA
---------------------------------------------
Rows: 50960
Unique races: 46
Duplicate lap keys: 0

PIT-STOP DATA
---------------------------------------------
Records: 1737
Unique races: 46
Missing raw durations: 0
Empty raw durations: 1
Unparsed durations: 1

SAVED FILES
---------------------------------------------
laps_all_2023_2024.csv
pit_stops_all_2023_2024.csv
fetch_qc_2023_2024.csv


In [6]:
import numpy as np
import pandas as pd

# 1. Did every newly fetched race return as many rows as the API reported?
mismatch = qc_df[qc_df["rows"] != qc_df["api_total"]]
print("Newly fetched races:", len(qc_df), "| row count != API total:", len(mismatch))
if len(mismatch):
    print(mismatch)

# 2. Per-race sanity: laps, drivers, rows
per_race = (laps_all.groupby(["season", "round"])
            .agg(rows=("lap", "size"), max_lap=("lap", "max"), drivers=("driver_id", "nunique"))
            .reset_index())
print("\nPer-race summary (min / median / max):")
print(per_race[["rows", "max_lap", "drivers"]].describe().loc[["min", "50%", "max"]].round(0))
print("\nRaces with fewer than 18 drivers or fewer than 40 laps:")
print(per_race[(per_race["drivers"] < 18) | (per_race["max_lap"] < 40)])

# 3. Pit durations: what do normal and very long stops look like?
dur = pits_all["duration_seconds"].dropna()
print("\nPit duration (s) percentiles:")
print(dur.quantile([0.01, 0.25, 0.5, 0.75, 0.95, 0.99]).round(1))
print("stops longer than 60 s:", int((dur > 60).sum()))
long_by_race = (pits_all[pits_all["duration_seconds"] > 60]
                .groupby(["season", "round"]).size().sort_values(ascending=False))
print("long stops by race:\n", long_by_race.head(8))

# 4. The one stop with no duration
print("\nStops with no parsed duration:")
print(pits_all[pits_all["duration_seconds"].isna()])

Newly fetched races: 36 | row count != API total: 0

Per-race summary (min / median / max):
       rows  max_lap  drivers
min   815.0     44.0     16.0
50%  1071.0     58.0     20.0
max  1513.0     78.0     20.0

Races with fewer than 18 drivers or fewer than 40 laps:
    season  round  rows  max_lap  drivers
19    2023     20  1106       71       17
29    2024      8  1233       78       16

Pit duration (s) percentiles:
0.01      17.4
0.25      22.1
0.50      23.9
0.75      28.8
0.95    1411.2
0.99    2394.5
Name: duration_seconds, dtype: float64
stops longer than 60 s: 159
long stops by race:
 season  round
2023    3        49
        19       18
2024    4        18
2023    13       17
        20       17
2024    21       17
        8        16
2023    16        1
dtype: int64

Stops with no parsed duration:
      season  round driver_id  lap  stop      time duration_raw  \
1327    2024     12   tsunoda   27     1  15:45:52                

      duration_seconds  
1327             

In [7]:
# CELL 5 — FINAL EXPANDED DATASET AUDIT

import pandas as pd
import numpy as np

PROCESSED_DIR = ROOT / "data" / "processed"

laps_all = pd.read_csv(
    PROCESSED_DIR / "laps_all_2023_2024.csv"
)

pits_all = pd.read_csv(
    PROCESSED_DIR / "pit_stops_all_2023_2024.csv"
)

qc = pd.read_csv(
    PROCESSED_DIR / "fetch_qc_2023_2024.csv"
)

# Expected race counts
expected_races = {
    2023: 22,
    2024: 24
}

print("RACE COVERAGE")
print("-" * 50)

for season, expected in expected_races.items():
    actual = laps_all[laps_all["season"] == season]["round"].nunique()

    print(
        f"{season}: {actual}/{expected} races",
        "| Status:",
        "OK" if actual == expected else "CHECK"
    )

# Check lap uniqueness
lap_duplicates = laps_all.duplicated(
    ["season", "round", "lap", "driver_id"]
).sum()

# Check pit-stop uniqueness
pit_duplicates = pits_all.duplicated(
    ["season", "round", "driver_id", "stop"]
).sum()

# Missing durations by season
missing_duration = pits_all[
    pits_all["duration_seconds"].isna()
].groupby("season").size()

print("\nDATA INTEGRITY")
print("-" * 50)
print("Lap rows:", len(laps_all))
print("Lap duplicates:", lap_duplicates)
print("Pit-stop rows:", len(pits_all))
print("Pit-stop duplicates:", pit_duplicates)
print("Missing parsed durations:", pits_all["duration_seconds"].isna().sum())

print("\nMISSING DURATIONS BY SEASON")
print(missing_duration)

print("\nFETCH QC")
print("-" * 50)
print("QC rows:", len(qc))
print("Mismatched race counts:", (qc["rows"] != qc["api_total"]).sum())

print("\nAUDIT COMPLETE")

RACE COVERAGE
--------------------------------------------------
2023: 22/22 races | Status: OK
2024: 24/24 races | Status: OK

DATA INTEGRITY
--------------------------------------------------
Lap rows: 50960
Lap duplicates: 0
Pit-stop rows: 1737
Pit-stop duplicates: 0
Missing parsed durations: 1

MISSING DURATIONS BY SEASON
season
2024    1
dtype: int64

FETCH QC
--------------------------------------------------
QC rows: 36
Mismatched race counts: 0

AUDIT COMPLETE


In [8]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r"D:\projects\f1-project")
PROCESSED = ROOT / "data" / "processed"

old_pits = pd.read_csv(
    PROCESSED / "pit_stops_2023_2024_first5.csv"
)

new_pits = pd.read_csv(
    PROCESSED / "pit_stops_all_2023_2024.csv"
)

print("OLD DATASET")
print("Shape:", old_pits.shape)
print("Columns:", old_pits.columns.tolist())

print("\nNEW DATASET")
print("Shape:", new_pits.shape)
print("Columns:", new_pits.columns.tolist())

OLD DATASET
Shape: (374, 7)
Columns: ['season', 'round', 'driver_id', 'lap', 'stop', 'time', 'duration']

NEW DATASET
Shape: (1737, 8)
Columns: ['season', 'round', 'driver_id', 'lap', 'stop', 'time', 'duration_raw', 'duration_seconds']


In [9]:

import pandas as pd
import numpy as np

# Columns that uniquely identify a pit stop
keys = ["season", "round", "driver_id", "lap", "stop"]

old = old_pits.copy()
new = new_pits.copy()

# Convert duration columns to numeric safely
old["old_duration"] = pd.to_numeric(
    old["duration"], errors="coerce"
)

new["new_duration"] = pd.to_numeric(
    new["duration_seconds"], errors="coerce"
)

# Keep only the columns needed for comparison
old_compare = old[keys + ["old_duration"]]
new_compare = new[keys + ["duration_raw", "new_duration"]]

# Match old and new records
comparison = old_compare.merge(
    new_compare,
    on=keys,
    how="outer",
    indicator=True
)

# Identify the different cases
matched = comparison[comparison["_merge"] == "both"].copy()

old_missing_new_valid = matched[
    matched["old_duration"].isna()
    & matched["new_duration"].notna()
]

old_valid_new_missing = matched[
    matched["old_duration"].notna()
    & matched["new_duration"].isna()
]

both_missing = matched[
    matched["old_duration"].isna()
    & matched["new_duration"].isna()
]

both_valid = matched[
    matched["old_duration"].notna()
    & matched["new_duration"].notna()
]

mismatches = both_valid[
    (both_valid["old_duration"] - both_valid["new_duration"]).abs() > 0.001
]

print("MATCHING SUMMARY")
print("Old records:", len(old))
print("New records:", len(new))
print("Matched records:", len(matched))
print("Old records not matched:", (comparison["_merge"] == "left_only").sum())
print("New records not matched:", (comparison["_merge"] == "right_only").sum())

print("\nDURATION COMPARISON")
print("Old missing, new valid:", len(old_missing_new_valid))
print("Old valid, new missing:", len(old_valid_new_missing))
print("Both missing:", len(both_missing))
print("Both valid:", len(both_valid))
print("Numeric mismatches (> 0.001 sec):", len(mismatches))

print("\nEXAMPLES: OLD MISSING, NEW VALID")
print(
    old_missing_new_valid[
        keys + ["old_duration", "duration_raw", "new_duration"]
    ].head(15).to_string(index=False)
)

print("\nEXAMPLES: NUMERIC MISMATCHES")
print(
    mismatches[
        keys + ["old_duration", "duration_raw", "new_duration"]
    ].head(10).to_string(index=False)
)

# Race-wise missing-duration comparison
race_summary = matched.groupby(
    ["season", "round"]
).agg(
    matched_stops=("stop", "size"),
    old_missing=("old_duration", lambda x: x.isna().sum()),
    new_missing=("new_duration", lambda x: x.isna().sum())
).reset_index()

print("\nRACE-WISE MISSING DURATION SUMMARY")
print(race_summary.to_string(index=False))


MATCHING SUMMARY
Old records: 374
New records: 1737
Matched records: 374
Old records not matched: 0
New records not matched: 1363

DURATION COMPARISON
Old missing, new valid: 70
Old valid, new missing: 0
Both missing: 0
Both valid: 304
Numeric mismatches (> 0.001 sec): 0

EXAMPLES: OLD MISSING, NEW VALID
 season  round driver_id  lap  stop  old_duration duration_raw  new_duration
   2023      3    alonso    8     1           NaN    15:26.940       926.940
   2023      3    alonso   55     2           NaN    14:34.306       874.306
   2023      3    alonso   57     3           NaN    30:45.073      1845.073
   2023      3    bottas    8     2           NaN    15:37.594       937.594
   2023      3    bottas   54     4           NaN    12:19.342       739.342
   2023      3    bottas   55     5           NaN     1:16.634        76.634
   2023      3    bottas   57     6           NaN    31:07.182      1867.182
   2023      3  de_vries    8     1           NaN    15:37.874       937.874
 

In [10]:

from pathlib import Path
import pandas as pd

ROOT = Path(r"D:\projects\f1-project")
PROCESSED = ROOT / "data" / "processed"

laps = pd.read_csv(
    PROCESSED / "laps_all_2023_2024.csv"
)

pits = pd.read_csv(
    PROCESSED / "pit_stops_all_2023_2024.csv"
)

print("EXPANDED LAP DATA")
print("Shape:", laps.shape)
print("Unique races:", laps[["season", "round"]].drop_duplicates().shape[0])
print("Duplicate lap keys:", laps.duplicated(
    ["season", "round", "driver_id", "lap"]
).sum())

print("\nEXPANDED PIT STOP DATA")
print("Shape:", pits.shape)
print("Unique races:", pits[["season", "round"]].drop_duplicates().shape[0])
print("Duplicate pit-stop keys:", pits.duplicated(
    ["season", "round", "driver_id", "lap", "stop"]
).sum())
print("Missing pit-stop durations:", pits["duration_seconds"].isna().sum())

print("\nSEASONS")
print("Lap data:", sorted(laps["season"].unique()))
print("Pit data:", sorted(pits["season"].unique()))


EXPANDED LAP DATA
Shape: (50960, 6)
Unique races: 46
Duplicate lap keys: 0

EXPANDED PIT STOP DATA
Shape: (1737, 8)
Unique races: 46
Duplicate pit-stop keys: 0
Missing pit-stop durations: 1

SEASONS
Lap data: [np.int64(2023), np.int64(2024)]
Pit data: [np.int64(2023), np.int64(2024)]


In [11]:
print(laps.columns.tolist())
print(laps.head(5).to_string(index=False))

print("\nPIT STOP COLUMNS")
print(pits.columns.tolist())
print(pits.head(5).to_string(index=False))

['season', 'round', 'lap', 'driver_id', 'position', 'lap_time']
 season  round  lap      driver_id  position lap_time
   2023      1    1 max_verstappen         1 1:39.019
   2023      1    1        leclerc         2 1:40.230
   2023      1    1          perez         3 1:41.063
   2023      1    1          sainz         4 1:41.659
   2023      1    1       hamilton         5 1:42.288

PIT STOP COLUMNS
['season', 'round', 'driver_id', 'lap', 'stop', 'time', 'duration_raw', 'duration_seconds']
 season  round driver_id  lap  stop     time duration_raw  duration_seconds
   2023      1     gasly    9     1 18:18:56       25.885            25.885
   2023      1    norris   10     1 18:20:31       32.766            32.766
   2023      1   tsunoda   10     1 18:20:34       25.267            25.267
   2023      1    bottas   11     1 18:22:06       25.399            25.399
   2023      1     albon   11     1 18:22:12       25.054            25.054


In [12]:

import pandas as pd
import numpy as np

# Work on copies; preserve original datasets
laps_work = laps.copy()
pits_work = pits.copy()

# Convert lap time from M:SS.mmm to seconds
def lap_time_to_seconds(value):
    if pd.isna(value):
        return np.nan

    parts = str(value).strip().split(":")

    if len(parts) == 2:
        return float(parts[0]) * 60 + float(parts[1])

    return np.nan

laps_work["lap_time_seconds"] = (
    laps_work["lap_time"].apply(lap_time_to_seconds)
)

# Remove implausible or unparseable lap times
laps_work = laps_work[
    laps_work["lap_time_seconds"].between(50, 200)
].copy()

# Identify pit laps
pit_keys = pits_work[
    ["season", "round", "driver_id", "lap"]
].drop_duplicates()

pit_keys["is_pit_lap"] = True

laps_work = laps_work.merge(
    pit_keys,
    on=["season", "round", "driver_id", "lap"],
    how="left"
)

laps_work["is_pit_lap"] = (
    laps_work["is_pit_lap"].fillna(False)
)

# Identify the lap immediately after a pit stop
pit_next_keys = pit_keys[
    ["season", "round", "driver_id", "lap"]
].copy()

pit_next_keys["lap"] = pit_next_keys["lap"] + 1
pit_next_keys["is_after_pit"] = True

laps_work = laps_work.merge(
    pit_next_keys,
    on=["season", "round", "driver_id", "lap"],
    how="left"
)

laps_work["is_after_pit"] = (
    laps_work["is_after_pit"].fillna(False)
)

# Eligible laps for reference pace and prediction
eligible = laps_work[
    (laps_work["lap"] > 1)
    & (~laps_work["is_pit_lap"])
    & (~laps_work["is_after_pit"])
].copy()

# Sort before calculating each driver's first five eligible laps
eligible = eligible.sort_values(
    ["season", "round", "driver_id", "lap"]
)

# Calculate reference pace from first five eligible laps
eligible["eligible_lap_number"] = (
    eligible.groupby(
        ["season", "round", "driver_id"]
    ).cumcount() + 1
)

reference_pace = (
    eligible[eligible["eligible_lap_number"] <= 5]
    .groupby(["season", "round", "driver_id"])
    ["lap_time_seconds"]
    .mean()
    .rename("reference_pace")
)

eligible = eligible.merge(
    reference_pace,
    on=["season", "round", "driver_id"],
    how="left"
)

# Keep prediction rows after the first five eligible laps
pace_data = eligible[
    eligible["eligible_lap_number"] > 5
].copy()

# Calculate target
pace_data["delta"] = (
    pace_data["lap_time_seconds"]
    - pace_data["reference_pace"]
)

# Basic dataset summary
print("EXPANDED PACE DATASET")
print("Rows:", len(pace_data))
print(
    "Races:",
    pace_data[["season", "round"]].drop_duplicates().shape[0]
)
print(
    "Driver-race groups:",
    pace_data[["season", "round", "driver_id"]]
    .drop_duplicates().shape[0]
)

print("\nTARGET SUMMARY")
print(pace_data["delta"].describe())

print("\nFIRST FIVE ROWS")
print(
    pace_data[
        [
            "season", "round", "driver_id",
            "lap", "lap_time_seconds",
            "reference_pace", "delta"
        ]
    ].head().to_string(index=False)
)



EXPANDED PACE DATASET
Rows: 42323
Races: 46
Driver-race groups: 881

TARGET SUMMARY
count    42323.000000
mean        -5.553323
std         14.100578
min        -55.966400
25%         -4.503300
50%         -1.683000
75%         -0.555100
max         78.977400
Name: delta, dtype: float64

FIRST FIVE ROWS
 season  round driver_id  lap  lap_time_seconds  reference_pace   delta
   2023      1     albon    7           100.646        100.3368  0.3092
   2023      1     albon    8           100.918        100.3368  0.5812
   2023      1     albon    9           100.969        100.3368  0.6322
   2023      1     albon   10           101.119        100.3368  0.7822
   2023      1     albon   13            98.649        100.3368 -1.6878


C:\Users\HP\AppData\Local\Temp\ipykernel_31604\862846408.py:43: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  laps_work["is_pit_lap"].fillna(False)
C:\Users\HP\AppData\Local\Temp\ipykernel_31604\862846408.py:61: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  laps_work["is_after_pit"].fillna(False)


In [13]:

# Start from the full expanded dataset
pace_data_expanded = pace_data.copy()

# Step 1: Exclude the two previously identified distorted races
distorted_races = [
    (2023, 3),
    (2024, 4)
]

race_mask = pd.Series(False, index=pace_data_expanded.index)

for season, rnd in distorted_races:
    race_mask |= (
        (pace_data_expanded["season"] == season)
        & (pace_data_expanded["round"] == rnd)
    )

pace_after_race_filter = pace_data_expanded[
    ~race_mask
].copy()

# Step 2: Identify suspicious race-lap groups
race_lap_stats = (
    pace_after_race_filter
    .groupby(["season", "round", "lap"])
    .agg(
        driver_count=("driver_id", "nunique"),
        median_delta=("delta", "median")
    )
    .reset_index()
)

flagged_groups = race_lap_stats[
    (race_lap_stats["driver_count"] >= 5)
    & (race_lap_stats["median_delta"].abs() > 5)
].copy()

# Step 3: Remove all rows belonging to flagged race-lap groups
flagged_keys = flagged_groups[
    ["season", "round", "lap"]
]

pace_after_lap_filter = pace_after_race_filter.merge(
    flagged_keys.assign(flagged=True),
    on=["season", "round", "lap"],
    how="left"
)

pace_after_lap_filter = pace_after_lap_filter[
    pace_after_lap_filter["flagged"].isna()
].drop(columns="flagged")

# Summary
print("FILTERING SUMMARY")
print("Original rows:", len(pace_data_expanded))
print("After distorted-race exclusion:", len(pace_after_race_filter))
print("Flagged race-lap groups:", len(flagged_groups))
print("Rows after race-lap exclusion:", len(pace_after_lap_filter))

print("\nFLAGGED RACE-LAP EXAMPLES")
print(flagged_groups.head(20).to_string(index=False))

print("\nTARGET DISTRIBUTION AFTER FILTERING")
print(pace_after_lap_filter["delta"].describe())



FILTERING SUMMARY
Original rows: 42323
After distorted-race exclusion: 40848
Flagged race-lap groups: 658
Rows after race-lap exclusion: 30258

FLAGGED RACE-LAP EXAMPLES
 season  round  lap  driver_count  median_delta
   2023      1   41            12       23.6536
   2023      2   17            13        9.8408
   2023      2   18             8       39.0463
   2023      2   19            11       34.0018
   2023      2   20            18       54.8144
   2023      4   10            15        5.7724
   2023      4   11            10       40.7008
   2023      4   12            11       27.5526
   2023      4   13            18       46.3107
   2023      6   52            15        7.5718
   2023      6   53            14       12.4793
   2023      6   54             8       18.8425
   2023      6   55             8       24.6699
   2023      6   56            14       22.4544
   2023      6   57            17       23.7416
   2023      6   58            18       23.8869
   2023      6

In [14]:

# Inspect the most extreme remaining observations
extreme_laps = pace_after_lap_filter.copy()

extreme_laps["abs_delta"] = extreme_laps["delta"].abs()

print("MOST EXTREME LAP DELTAS")
print(
    extreme_laps.sort_values(
        "abs_delta", ascending=False
    )[
        [
            "season", "round", "driver_id", "lap",
            "lap_time_seconds", "reference_pace", "delta"
        ]
    ].head(20).to_string(index=False)
)

print("\nEXTREME VALUE COUNTS")
for threshold in [5, 10, 15, 20]:
    count = (extreme_laps["abs_delta"] > threshold).sum()
    print(f"Rows with |delta| > {threshold}s: {count}")

# Check how many drivers were present in each extreme race-lap group
extreme_groups = (
    extreme_laps.groupby(["season", "round", "lap"])
    .agg(
        driver_count=("driver_id", "nunique"),
        median_delta=("delta", "median"),
        min_delta=("delta", "min"),
        max_delta=("delta", "max")
    )
    .reset_index()
)

print("\nRACE-LAP GROUPS CONTAINING EXTREME VALUES")
print(
    extreme_groups[
        (extreme_groups["min_delta"] < -10)
        | (extreme_groups["max_delta"] > 10)
    ].sort_values(
        "max_delta", ascending=False
    ).head(20).to_string(index=False)
)


MOST EXTREME LAP DELTAS
 season  round      driver_id  lap  lap_time_seconds  reference_pace    delta
   2024      2         norris    8           174.005         95.0276  78.9774
   2024      2       hamilton    8           173.631         95.3162  78.3148
   2024      2     hulkenberg    8           170.478         96.5140  73.9640
   2023     15 max_verstappen   21           172.082        101.3740  70.7080
   2024      2           zhou    8           166.499         97.1214  69.3776
   2023     10          gasly   46           155.981         93.9332  62.0478
   2023     15          perez   21           162.641        101.8926  60.7484
   2023      8       de_vries   35           139.671         79.4556  60.2154
   2023     15         bottas   21           158.073        102.0022  56.0708
   2023      2          albon   27           151.949         97.4012  54.5478
   2024     11         norris   64           120.257         70.2966  49.9604
   2024      9 max_verstappen   55      

In [15]:

# Keep the complete filtered dataset unchanged
pace_all_filtered = pace_after_lap_filter.copy()

# Identify extreme individual deviations
extreme_mask = pace_all_filtered["delta"].abs() > 10

# Create a separate clean-pace candidate
pace_clean_candidate = pace_all_filtered[
    ~extreme_mask
].copy()

print("SENSITIVITY ANALYSIS")
print("Rows before individual outlier filter:", len(pace_all_filtered))
print("Extreme observations (|delta| > 10):", extreme_mask.sum())
print("Percentage excluded:", round(extreme_mask.mean() * 100, 2), "%")
print("Rows in clean-pace candidate:", len(pace_clean_candidate))

print("\nDISTRIBUTION BEFORE INDIVIDUAL OUTLIER FILTER")
print(pace_all_filtered["delta"].describe())

print("\nDISTRIBUTION AFTER INDIVIDUAL OUTLIER FILTER")
print(pace_clean_candidate["delta"].describe())

print("\nEXTREME OBSERVATIONS BY SEASON")
print(
    pace_all_filtered.loc[extreme_mask]
    .groupby("season")
    .size()
    .to_string()
)

print("\nEXTREME OBSERVATIONS BY RACE")
print(
    pace_all_filtered.loc[extreme_mask]
    .groupby(["season", "round"])
    .size()
    .sort_values(ascending=False)
    .head(15)
    .to_string()
)


SENSITIVITY ANALYSIS
Rows before individual outlier filter: 30258
Extreme observations (|delta| > 10): 204
Percentage excluded: 0.67 %
Rows in clean-pace candidate: 30054

DISTRIBUTION BEFORE INDIVIDUAL OUTLIER FILTER
count    30258.000000
mean        -1.281133
std          2.392321
min        -32.953800
25%         -2.133350
50%         -1.236000
75%         -0.413600
max         78.977400
Name: delta, dtype: float64

DISTRIBUTION AFTER INDIVIDUAL OUTLIER FILTER
count    30054.000000
mean        -1.302217
std          1.480977
min         -9.968800
25%         -2.124000
50%         -1.235000
75%         -0.418250
max          9.723400
Name: delta, dtype: float64

EXTREME OBSERVATIONS BY SEASON
season
2023    145
2024     59

EXTREME OBSERVATIONS BY RACE
season  round
2023    20       67
        13       53
2024    12       17
        9        12
2023    15        9
2024    2         8
        11        7
2023    1         6
2024    21        4
2023    6         3
2024    23        3
 

In [16]:

# Create independent copies
pace_all_filtered = pace_all_filtered.copy()
pace_clean_candidate = pace_clean_candidate.copy()

# Calculate stint_lap using actual pit-stop laps
def add_stint_lap(df, pit_data):
    result = df.copy()

    # Map each driver-race to its pit-stop laps
    pit_map = (
        pit_data.groupby(["season", "round", "driver_id"])["lap"]
        .apply(lambda x: sorted(x.dropna().astype(int).unique()))
        .to_dict()
    )

    def calculate_stint_lap(row):
        key = (row["season"], row["round"], row["driver_id"])
        current_lap = int(row["lap"])
        previous_stops = [
            stop_lap for stop_lap in pit_map.get(key, [])
            if stop_lap < current_lap
        ]

        previous_pit_lap = max(previous_stops) if previous_stops else 0

        return current_lap - previous_pit_lap

    result["stint_lap"] = result.apply(
        calculate_stint_lap, axis=1
    )

    return result

pace_all_filtered = add_stint_lap(pace_all_filtered, pits)
pace_clean_candidate = add_stint_lap(pace_clean_candidate, pits)

# Save both datasets
all_path = PROCESSED / "pace_all_filtered_2023_2024.csv"
clean_path = PROCESSED / "pace_clean_candidate_2023_2024.csv"

pace_all_filtered.to_csv(all_path, index=False)
pace_clean_candidate.to_csv(clean_path, index=False)

print("DATASETS SAVED")
print(all_path)
print(clean_path)

print("\nDATASET SHAPES")
print("All filtered:", pace_all_filtered.shape)
print("Clean candidate:", pace_clean_candidate.shape)

print("\nSTINT LAP CHECK")
print(pace_clean_candidate["stint_lap"].describe())

print("\nMISSING VALUES IN MODEL FEATURES")
print(
    pace_clean_candidate[
        ["lap", "stint_lap", "reference_pace", "delta"]
    ].isna().sum()
)


DATASETS SAVED
D:\projects\f1-project\data\processed\pace_all_filtered_2023_2024.csv
D:\projects\f1-project\data\processed\pace_clean_candidate_2023_2024.csv

DATASET SHAPES
All filtered: (30258, 13)
Clean candidate: (30054, 13)

STINT LAP CHECK
count    30054.000000
mean        15.835895
std          9.801074
min          2.000000
25%          8.000000
50%         14.000000
75%         21.000000
max         58.000000
Name: stint_lap, dtype: float64

MISSING VALUES IN MODEL FEATURES
lap               0
stint_lap         0
reference_pace    0
delta             0
dtype: int64


In [17]:

import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Load both datasets
all_data = pd.read_csv(
    PROCESSED / "pace_all_filtered_2023_2024.csv"
)

clean_data = pd.read_csv(
    PROCESSED / "pace_clean_candidate_2023_2024.csv"
)

def evaluate_models(data, dataset_name):

    features = ["lap", "stint_lap"]
    target = "delta"

    X = data[features]
    y = data[target]

    # Each season-round combination is one validation group
    groups = (
        data["season"].astype(str)
        + "_R"
        + data["round"].astype(str)
    )

    logo = LeaveOneGroupOut()

    # Store predictions from every held-out race
    predictions = {
        "Zero baseline": np.full(len(data), np.nan),
        "Training-mean baseline": np.full(len(data), np.nan),
        "Ridge": np.full(len(data), np.nan),
        "Random Forest": np.full(len(data), np.nan)
    }

    for train_idx, test_idx in logo.split(X, y, groups):

        X_train = X.iloc[train_idx]
        X_test = X.iloc[test_idx]

        y_train = y.iloc[train_idx]

        # Baseline 1: predict no change from reference pace
        predictions["Zero baseline"][test_idx] = 0.0

        # Baseline 2: predict the training target mean
        predictions["Training-mean baseline"][test_idx] = (
            y_train.mean()
        )

        # Ridge Regression
        ridge = Pipeline([
            ("scaler", StandardScaler()),
            ("model", Ridge(alpha=1))
        ])

        ridge.fit(X_train, y_train)

        predictions["Ridge"][test_idx] = ridge.predict(X_test)

        # Random Forest
        rf = RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=10,
            max_features=1.0,
            random_state=42,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)

        predictions["Random Forest"][test_idx] = (
            rf.predict(X_test)
        )

    # Calculate pooled out-of-fold metrics
    results = []

    for model_name, pred in predictions.items():

        mae = mean_absolute_error(y, pred)

        rmse = np.sqrt(mean_squared_error(y, pred))

        results.append({
            "Dataset": dataset_name,
            "Model": model_name,
            "MAE": mae,
            "RMSE": rmse
        })

    return pd.DataFrame(results)


# Evaluate both datasets
results_all = evaluate_models(
    all_data, "All filtered"
)

results_clean = evaluate_models(
    clean_data, "Clean candidate"
)

final_results = pd.concat(
    [results_all, results_clean],
    ignore_index=True
)

print("LEAVE-ONE-RACE-OUT MODEL RESULTS")
print(
    final_results.to_string(
        index=False,
        formatters={
            "MAE": "{:.4f}".format,
            "RMSE": "{:.4f}".format
        }
    )
)


LEAVE-ONE-RACE-OUT MODEL RESULTS
        Dataset                  Model    MAE   RMSE
   All filtered          Zero baseline 1.6592 2.7137
   All filtered Training-mean baseline 1.2068 2.3999
   All filtered                  Ridge 1.0728 2.3460
   All filtered          Random Forest 1.1461 2.4236
Clean candidate          Zero baseline 1.5430 1.9721
Clean candidate Training-mean baseline 1.0893 1.4910
Clean candidate                  Ridge 0.9592 1.4194
Clean candidate          Random Forest 0.9979 1.4928


In [18]:

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
import pandas as pd

data = clean_data.copy()

X = data[["lap", "stint_lap"]]
y = data["delta"]

groups = (
    data["season"].astype(str)
    + "_R"
    + data["round"].astype(str)
)

logo = LeaveOneGroupOut()
race_results = []

for train_idx, test_idx in logo.split(X, y, groups):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    # Training-mean baseline
    mean_prediction = [y_train.mean()] * len(test_idx)

    # Ridge model
    ridge = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1))
    ])

    ridge.fit(X_train, y_train)
    ridge_prediction = ridge.predict(X_test)

    race_results.append({
        "Race": groups.iloc[test_idx[0]],
        "Test rows": len(test_idx),
        "Baseline MAE": mean_absolute_error(
            y_test, mean_prediction
        ),
        "Ridge MAE": mean_absolute_error(
            y_test, ridge_prediction
        )
    })

race_results = pd.DataFrame(race_results)

race_results["MAE improvement (%)"] = (
    (race_results["Baseline MAE"] - race_results["Ridge MAE"])
    / race_results["Baseline MAE"] * 100
)

print("RACE-WISE VALIDATION")
print(
    race_results.sort_values("Ridge MAE")
    .to_string(index=False, formatters={
        "Baseline MAE": "{:.3f}".format,
        "Ridge MAE": "{:.3f}".format,
        "MAE improvement (%)": "{:.1f}".format
    })
)

print("\nSUMMARY")
print("Races where Ridge beats baseline:",
      (race_results["MAE improvement (%)"] > 0).sum(),
      "out of", len(race_results))

print("Median race-wise Ridge MAE:",
      round(race_results["Ridge MAE"].median(), 4))

print("Worst race by Ridge MAE:")
print(
    race_results.loc[
        race_results["Ridge MAE"].idxmax()
    ].to_string()
)


RACE-WISE VALIDATION
    Race  Test rows Baseline MAE Ridge MAE MAE improvement (%)
2023_R10        701        0.767     0.475                38.1
 2023_R5        978        0.698     0.480                31.1
2023_R22        963        0.792     0.543                31.4
 2024_R6        844        0.851     0.545                36.0
 2024_R3        775        0.859     0.565                34.2
2024_R16        827        0.757     0.566                25.2
2023_R11       1068        0.847     0.583                31.2
 2023_R8       1036        0.792     0.593                25.2
 2023_R7       1107        0.967     0.618                36.1
 2024_R5        661        0.819     0.625                23.7
 2024_R1        924        0.900     0.636                29.3
2024_R15       1254        0.776     0.662                14.7
2024_R10       1106        0.920     0.685                25.5
2024_R18       1010        0.891     0.715                19.8
2023_R18        821        0.941  

In [19]:

def race_set(df):
    return set(
        zip(df["season"].astype(int), df["round"].astype(int))
    )

all_races = race_set(laps)
after_race_exclusion = race_set(pace_after_race_filter)
after_group_filter = race_set(pace_after_lap_filter)
clean_races = race_set(pace_clean_candidate)

print("RACE COVERAGE AUDIT")
print("Races in original lap data:", len(all_races))
print("After explicit race exclusions:", len(after_race_exclusion))
print("After race-lap group filtering:", len(after_group_filter))
print("In final clean candidate:", len(clean_races))

print("\nRACES LOST DURING RACE-LAP FILTERING")
for race in sorted(after_race_exclusion - after_group_filter):
    print(f"{race[0]} Round {race[1]}")

print("\nRACES PRESENT IN CLEAN CANDIDATE")
print(sorted(clean_races))

print("\nROWS PER RACE IN CLEAN CANDIDATE")
race_counts = (
    pace_clean_candidate.groupby(["season", "round"])
    .size()
    .reset_index(name="rows")
    .sort_values("rows")
)
print(race_counts.to_string(index=False))

print("\nRACES WITH FEWER THAN 200 ROWS")
print(
    race_counts[race_counts["rows"] < 200].to_string(index=False)
)


RACE COVERAGE AUDIT
Races in original lap data: 46
After explicit race exclusions: 44
After race-lap group filtering: 38
In final clean candidate: 38

RACES LOST DURING RACE-LAP FILTERING
2023 Round 16
2023 Round 17
2023 Round 21
2024 Round 19
2024 Round 20
2024 Round 24

RACES PRESENT IN CLEAN CANDIDATE
[(2023, 1), (2023, 2), (2023, 4), (2023, 5), (2023, 6), (2023, 7), (2023, 8), (2023, 9), (2023, 10), (2023, 11), (2023, 12), (2023, 13), (2023, 14), (2023, 15), (2023, 18), (2023, 19), (2023, 20), (2023, 22), (2024, 1), (2024, 2), (2024, 3), (2024, 5), (2024, 6), (2024, 7), (2024, 8), (2024, 9), (2024, 10), (2024, 11), (2024, 12), (2024, 13), (2024, 14), (2024, 15), (2024, 16), (2024, 17), (2024, 18), (2024, 21), (2024, 22), (2024, 23)]

ROWS PER RACE IN CLEAN CANDIDATE
 season  round  rows
   2024     23     1
   2024      9    52
   2023     13   100
   2023     20   322
   2023     19   528
   2024     12   546
   2023     12   627
   2024     14   654
   2024      5   661
   2023  

In [20]:

# Keep races with at least 200 prediction rows
minimum_rows = 200

race_counts = (
    clean_data.groupby(["season", "round"])
    .size()
    .reset_index(name="rows")
)

eligible_races = race_counts[
    race_counts["rows"] >= minimum_rows
][["season", "round"]]

robust_eval_data = clean_data.merge(
    eligible_races,
    on=["season", "round"],
    how="inner"
).copy()

print("ROBUST EVALUATION SUBSET")
print("Minimum rows per race:", minimum_rows)
print("Races retained:", len(eligible_races))
print("Rows retained:", len(robust_eval_data))
print("Races excluded for low row count:",
      len(race_counts) - len(eligible_races))

print("\nRACE COUNTS")
print(
    robust_eval_data.groupby(["season", "round"])
    .size()
    .to_string()
)

# Evaluate with leave-one-race-out validation
X = robust_eval_data[["lap", "stint_lap"]]
y = robust_eval_data["delta"]

groups = (
    robust_eval_data["season"].astype(str)
    + "_R"
    + robust_eval_data["round"].astype(str)
)

logo = LeaveOneGroupOut()

predictions_ridge = np.full(len(robust_eval_data), np.nan)
predictions_mean = np.full(len(robust_eval_data), np.nan)

for train_idx, test_idx in logo.split(X, y, groups):

    ridge = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1))
    ])

    ridge.fit(X.iloc[train_idx], y.iloc[train_idx])

    predictions_ridge[test_idx] = ridge.predict(
        X.iloc[test_idx]
    )

    predictions_mean[test_idx] = y.iloc[train_idx].mean()

print("\nROBUST SUBSET RESULTS")
print(
    "Training-mean MAE:",
    round(mean_absolute_error(y, predictions_mean), 4)
)
print(
    "Ridge MAE:",
    round(mean_absolute_error(y, predictions_ridge), 4)
)
print(
    "Training-mean RMSE:",
    round(np.sqrt(mean_squared_error(y, predictions_mean)), 4)
)
print(
    "Ridge RMSE:",
    round(np.sqrt(mean_squared_error(y, predictions_ridge)), 4)
)


ROBUST EVALUATION SUBSET
Minimum rows per race: 200
Races retained: 35
Rows retained: 29901
Races excluded for low row count: 3

RACE COUNTS
season  round
2023    1         817
        2         724
        4         741
        5         978
        6         872
        7        1107
        8        1036
        9        1093
        10        701
        11       1068
        12        627
        14        790
        15        860
        18        821
        19        528
        20        322
        22        963
2024    1         924
        2         722
        3         775
        5         661
        6         844
        7        1065
        8         789
        10       1106
        11       1172
        12        546
        13       1155
        14        654
        15       1254
        16        827
        17        780
        18       1010
        21        829
        22        740

ROBUST SUBSET RESULTS
Training-mean MAE: 1.0803
Ridge MAE: 0.9491
Training

In [21]:

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
import numpy as np
import pandas as pd

data = robust_eval_data.copy()

X = data[["lap", "stint_lap"]]
y = data["delta"]

groups = (
    data["season"].astype(str)
    + "_R"
    + data["round"].astype(str)
)

logo = LeaveOneGroupOut()
robust_race_results = []

for train_idx, test_idx in logo.split(X, y, groups):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    baseline_prediction = np.full(
        len(test_idx), y_train.mean()
    )

    ridge = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1))
    ])

    ridge.fit(X_train, y_train)
    ridge_prediction = ridge.predict(X_test)

    baseline_mae = mean_absolute_error(
        y_test, baseline_prediction
    )
    ridge_mae = mean_absolute_error(
        y_test, ridge_prediction
    )

    robust_race_results.append({
        "Race": groups.iloc[test_idx[0]],
        "Test rows": len(test_idx),
        "Baseline MAE": baseline_mae,
        "Ridge MAE": ridge_mae,
        "Ridge beats baseline": ridge_mae < baseline_mae
    })

robust_race_results = pd.DataFrame(robust_race_results)

print("ROBUST RACE-WISE VALIDATION")
print(
    robust_race_results.to_string(
        index=False,
        formatters={
            "Baseline MAE": "{:.4f}".format,
            "Ridge MAE": "{:.4f}".format
        }
    )
)

print("\nSUMMARY")
print(
    "Ridge beats baseline:",
    robust_race_results["Ridge beats baseline"].sum(),
    "out of",
    len(robust_race_results)
)

print(
    "Median race-wise Ridge MAE:",
    round(robust_race_results["Ridge MAE"].median(), 4)
)

# Save the race-wise validation results
output_path = PROCESSED / "ridge_robust_racewise_results.csv"
robust_race_results.to_csv(output_path, index=False)

print("\nSaved:", output_path)


ROBUST RACE-WISE VALIDATION
    Race  Test rows Baseline MAE Ridge MAE  Ridge beats baseline
 2023_R1        817       1.0621    0.8439                  True
2023_R10        701       0.7672    0.4713                  True
2023_R11       1068       0.8474    0.5800                  True
2023_R12        627       1.3467    1.1976                  True
2023_R14        790       1.1255    0.9914                  True
2023_R15        860       1.0281    0.9020                  True
2023_R18        821       0.9416    0.7264                  True
2023_R19        528       3.0455    3.4351                 False
 2023_R2        724       0.9948    0.8663                  True
2023_R20        322       3.2183    3.1740                  True
2023_R22        963       0.7914    0.5389                  True
 2023_R4        741       0.7972    0.7541                  True
 2023_R5        978       0.6977    0.4783                  True
 2023_R6        872       1.1431    1.1715                 Fal

In [22]:

from pathlib import Path

# Save the complete model comparison
comparison_path = PROCESSED / "model_comparison_2023_2024.csv"
final_results.to_csv(comparison_path, index=False)

# Save the 38-race validation results too
full_race_path = PROCESSED / "ridge_38_race_validation.csv"
race_results.to_csv(full_race_path, index=False)

# DECISIONS.md is in the project root
decisions_path = ROOT / "DECISIONS.md"

day5_section = """
## Day 5 — Expanded data and model validation (2023–2024)

### Data collection and quality

- Collected lap and pit-stop records across 46 races from the 2023 and 2024 seasons.
- Lap dataset: 50,960 rows; zero duplicate race-driver-lap keys.
- Pit-stop dataset: 1,737 records; zero duplicate race-driver-stop keys.
- One pit-stop duration remained missing and was preserved as NaN.
- Reconciled all 374 pit stops from the original first-10-race dataset against the expanded dataset.
- The 70 previously missing durations were parsing failures for m:ss.mmm values. All 374 matched records agreed after correct parsing.

### Target construction

- Converted lap times to seconds.
- Excluded lap 1, pit laps, and the lap immediately following a pit stop.
- Calculated each driver's reference pace from their first five eligible laps.
- Target: delta = lap_time_seconds - reference_pace.
- Created stint_lap from recorded pit-stop laps. This is a pit-stop-derived proxy, not verified tyre age.

### Retrospective filtering

- Initial prediction rows: 42,323 across 46 races.
- Explicitly excluded 2023 R3 and 2024 R4 based on the earlier distortion analysis.
- Excluded race-lap groups with at least five drivers and absolute median delta greater than 5 seconds.
- This removed 10,590 rows and eliminated six additional races entirely: 2023 R16, R17, R21 and 2024 R19, R20, R24.
- Remaining rows after race-lap filtering: 30,258 across 38 races.
- Additionally excluded 204 individual observations with absolute delta greater than 10 seconds (0.67%).
- Final clean-pace candidate: 30,054 rows across 38 races.
- Preserved both the all-filtered and clean-candidate datasets.

These exclusions use completed-race information and are retrospective. They define a normal-pace research subset, not an unbiased prospective sample of all race conditions.

### Model comparison

Features: lap and stint_lap.

Validation: Leave-One-Race-Out (LORO), with race as the held-out group.

| Dataset | Model | MAE (s) | RMSE (s) |
|---|---|---:|---:|
| All filtered | Zero baseline | 1.6592 | 2.7137 |
| All filtered | Training-mean baseline | 1.2068 | 2.3999 |
| All filtered | Ridge | 1.0728 | 2.3460 |
| All filtered | Random Forest | 1.1461 | 2.4236 |
| Clean candidate | Zero baseline | 1.5430 | 1.9721 |
| Clean candidate | Training-mean baseline | 1.0893 | 1.4910 |
| Clean candidate | Ridge | 0.9592 | 1.4194 |
| Clean candidate | Random Forest | 0.9979 | 1.4928 |

Ridge was the best-performing model among those tested on both datasets.

### Robust evaluation

- Required at least 200 prediction rows per race.
- Retained 35 races and 29,901 rows.
- Training-mean baseline: MAE 1.0803 s, RMSE 1.4721 s.
- Ridge: MAE 0.9491 s, RMSE 1.3983 s.
- Ridge reduced pooled MAE by approximately 12.1% and RMSE by approximately 5.0%.
- Ridge beat the training-mean baseline in 31 of 35 held-out races (88.6%).
- Median race-wise Ridge MAE: 0.8371 s.
- Ridge performed worse than the baseline in 2023 R19, 2023 R6, 2024 R11 and 2024 R8.

### Limitations and interpretation

- The model was evaluated on 38 retained races, not all 46 collected races.
- The 35-race robust subset is selected retrospectively by row count.
- Filtering based on completed-race target values makes the clean-pace results optimistic for prospective deployment.
- Only two model features were tested: lap and stint_lap.
- stint_lap is not verified tyre age.
- Model coefficients and predictions are associations within this dataset; they do not directly measure fuel burn or tyre degradation.
- Jolpica data does not provide verified tyre compounds, tyre age, weather, fuel load, or total pit-lane loss.
- These results establish a clean-pace baseline, not a complete race-strategy optimizer.

### Day 5 decision

Use Ridge Regression with lap and stint_lap as the first baseline model for the next project stage. Keep Random Forest as a comparison model. Preserve the filtering limitations and race-level failures in future reporting.

"""

marker = "## Day 5 — Expanded data and model validation (2023–2024)"

if decisions_path.exists():
    existing_text = decisions_path.read_text(encoding="utf-8")
else:
    existing_text = "# Project Decisions\n"

if marker not in existing_text:
    with decisions_path.open("a", encoding="utf-8") as f:
        f.write("\n" + day5_section)
    print("Day 5 section appended to DECISIONS.md")
else:
    print("Day 5 section already exists; no duplicate was added.")

print("\nSAVED FILES")
print(comparison_path)
print(full_race_path)
print(decisions_path)


Day 5 section appended to DECISIONS.md

SAVED FILES
D:\projects\f1-project\data\processed\model_comparison_2023_2024.csv
D:\projects\f1-project\data\processed\ridge_38_race_validation.csv
D:\projects\f1-project\DECISIONS.md


In [23]:

from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# 1. Locate project and load the clean candidate dataset
ROOT = Path.cwd()

if ROOT.name == "notebooks":
    ROOT = ROOT.parent

PROCESSED = ROOT / "data" / "processed"

pace = pd.read_csv(
    PROCESSED / "pace_clean_candidate_2023_2024.csv"
)

# 2. Create a unique race identifier
pace["race_id"] = (
    pace["season"].astype(str) + "_R" +
    pace["round"].astype(str)
)

# 3. Keep races with at least 200 clean observations
race_counts = pace.groupby("race_id").size()
eligible_races = race_counts[race_counts >= 200].index

data = pace[pace["race_id"].isin(eligible_races)].copy()

print("Eligible races:", data["race_id"].nunique())
print("Rows:", len(data))

# 4. Define target and features
y = data["delta"].to_numpy()
groups = data["race_id"].to_numpy()

X_linear = data[["lap", "stint_lap"]].copy()

X_quadratic = X_linear.copy()
X_quadratic["stint_lap_sq"] = X_quadratic["stint_lap"] ** 2

# 5. Define models
models = {
    "Training mean baseline": None,

    "Ridge linear": make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    ),

    "Ridge quadratic": make_pipeline(
        StandardScaler(),
        Ridge(alpha=1.0)
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=10,
        max_features=1.0,
        random_state=42,
        n_jobs=-1
    )
}

features = {
    "Training mean baseline": None,
    "Ridge linear": X_linear,
    "Ridge quadratic": X_quadratic,
    "Random Forest": X_linear
}

# 6. Leave one entire race out at a time
logo = LeaveOneGroupOut()
results = []

for model_name, model in models.items():

    actual_all = []
    predicted_all = []

    for train_idx, test_idx in logo.split(X_linear, y, groups):

        y_train = y[train_idx]
        y_test = y[test_idx]

        if model is None:
            predictions = np.full(
                len(test_idx),
                y_train.mean()
            )
        else:
            X_model = features[model_name]

            model.fit(
                X_model.iloc[train_idx],
                y_train
            )

            predictions = model.predict(
                X_model.iloc[test_idx]
            )

        actual_all.extend(y_test)
        predicted_all.extend(predictions)

    mae = mean_absolute_error(
        actual_all,
        predicted_all
    )

    rmse = np.sqrt(
        mean_squared_error(actual_all, predicted_all)
    )

    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse
    })

# 7. Display comparison
comparison = pd.DataFrame(results)

print("\nMODEL COMPARISON — SAME 35 RACE FOLDS")
display(comparison.round(4))


Eligible races: 35
Rows: 29901

MODEL COMPARISON — SAME 35 RACE FOLDS


,Model,MAE,RMSE
0,Training mean baseline,1.0803,1.4721
1,Ridge linear,0.9491,1.3983
2,Ridge quadratic,0.9405,1.3887
3,Random Forest,0.9855,1.4679


In [24]:

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

# Prepare quadratic features
X_final = data[["lap", "stint_lap"]].copy()
X_final["stint_lap_sq"] = X_final["stint_lap"] ** 2

y_final = data["delta"]

# Fit the selected model on all eligible data
final_model = make_pipeline(
    StandardScaler(),
    Ridge(alpha=1.0)
)

final_model.fit(X_final, y_final)

# Extract scaler and Ridge components
scaler = final_model.named_steps["standardscaler"]
ridge = final_model.named_steps["ridge"]

# Convert standardized coefficients to original feature units
coefficients = ridge.coef_ / scaler.scale_

# Adjust intercept to original feature units
intercept = ridge.intercept_ - np.sum(
    ridge.coef_ * scaler.mean_ / scaler.scale_
)

coefficient_table = pd.DataFrame({
    "Feature": ["lap", "stint_lap", "stint_lap_sq"],
    "Coefficient": coefficients
})

print("FINAL QUADRATIC RIDGE MODEL")
print(f"Intercept: {intercept:.6f}")

display(coefficient_table.round(6))

print("\nMODEL EQUATION:")
print(
    f"delta = {intercept:.6f}"
    f" + ({coefficients[0]:.6f} × lap)"
    f" + ({coefficients[1]:.6f} × stint_lap)"
    f" + ({coefficients[2]:.6f} × stint_lap²)"
)


FINAL QUADRATIC RIDGE MODEL
Intercept: -0.848157


,Feature,Coefficient
0,lap,-0.035306
1,stint_lap,0.077536
2,stint_lap_sq,-0.001460



MODEL EQUATION:
delta = -0.848157 + (-0.035306 × lap) + (0.077536 × stint_lap) + (-0.001460 × stint_lap²)
